
Run SAM server on Colab

Important: keep cells separate because '%%writefile app.py' writes its contents out to the app.py file.



In [1]:
!pip -q install fastapi uvicorn python-multipart

In [ ]:
import torch
import torchvision
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())
import sys
!{sys.executable} -m pip -q install opencv-python matplotlib scikit-learn
!{sys.executable} -m pip -q install 'git+https://github.com/facebookresearch/sam3.git'


In [ ]:
import time
import torch

import cv2
import numpy as np
from PIL import Image

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

In [ ]:
%%writefile app.py
from fastapi import FastAPI, UploadFile, File, Form
from PIL import Image
import io
import torch

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

app = FastAPI()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_sam3_image_model(device=device)
processor = Sam3Processor(model)

@app.post("/infer")
async def infer(
    file: UploadFile = File(...),
    prompt: str = Form("posters, flyers")
):
    data = await file.read()
    image = Image.open(io.BytesIO(data)).convert("RGB")

    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        state = processor.set_image(image)
        output = processor.set_text_prompt(
            state=state,
            prompt=prompt
        )

    boxes = output["boxes"]
    scores = output["scores"]

    # Move tensors to CPU + JSON-safe format
    boxes = boxes.detach().cpu().tolist()
    scores = scores.detach().cpu().tolist()

    results = []
    for box, score in zip(boxes, scores):
        x1, y1, x2, y2 = box
        results.append({
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "score": float(score),
            "label": prompt
        })

    return {
        "status": "ok",
        "device": device,
        "prompt": prompt,
        "num_boxes": len(results),
        "boxes": results
    }

In [ ]:
!pkill -f "uvicorn app:app" || true
!uvicorn app:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!ls -lh cloudflared

!./cloudflared tunnel --url http://localhost:8000